# 02 — Preprocessing

**Input:** `data/raw/M1_data.csv` (includes m1_purchase target field)  
**Output:** `data/processed/processed_m1_data.csv` (21 features scaled + target)

Steps:
1. Load M1 dataset
2. Separate target variable (`m1_purchase`) from features
3. Impute missing values
4. Outlier detection & removal (IQR method)
5. Encode categoricals
6. Feature scaling (StandardScaler) — applied ONLY to features, not target


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PLOT_DIR = Path('../plots/preprocessing')
PLOT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv('../data/raw/M1_data.csv')
print('Raw shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head()

## Step 1:  Drop Unused Columns

In [ ]:
# No ID column in M1 dataset, but drop if exists
df = df.drop(columns=['ID'], errors='ignore')
print('After drop unused columns:', df.shape)

## Step 1b: Separate Target Variable

In [ ]:
# Separate target variable (m1_purchase) from features
# m1_purchase should NOT be scaled — it's a classification target
if 'm1_purchase' in df.columns:
    y = df['m1_purchase'].copy()  # Save target variable
    df_features = df.drop(columns=['m1_purchase'])
    print(f"Target (m1_purchase) separated.")
    print(f"  Distribution: {y.value_counts().to_dict()}")
else:
    print("Warning: m1_purchase column not found. Proceeding without target variable.")

## Step 2: Handle Missing Values

In [ ]:
print('Missing before imputation:')
print(df_features.isnull().sum()[df_features.isnull().sum() > 0])

# Numeric: fill with median
num_cols = df_features.select_dtypes(include='number').columns.tolist()
for col in num_cols:
    df_features[col].fillna(df_features[col].median(), inplace=True)

# Categorical: fill with mode
cat_cols = df_features.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    df_features[col].fillna(df_features[col].mode()[0], inplace=True)

print('\nMissing after imputation:', df_features.isnull().sum().sum())

## Step 3 : Outlier Detection & Removal (IQR)

In [ ]:
# --- Boxplots BEFORE handling ---
numeric_cols = df_features.select_dtypes(include='number').columns.tolist()
if len(numeric_cols) >= 3:
    sample_cols = numeric_cols[:3]
else:
    sample_cols = numeric_cols

fig, axes = plt.subplots(1, len(sample_cols), figsize=(15, 4))
if len(sample_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, sample_cols):
    sns.boxplot(y=df_features[col], ax=ax, color='steelblue')
    ax.set_title(f'{col} — Before')
plt.suptitle('Boxplots Before Outlier Handling')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'boxplots_before_outlier_handling.png', dpi=150)
plt.show()

In [ ]:
# IQR Capping (Winsorization) — cap outliers at 1.5*IQR fences
outlier_cols = df_features.select_dtypes(include='number').columns.tolist()

for col in outlier_cols:
    Q1  = df_features[col].quantile(0.25)
    Q3  = df_features[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df_features[col] < lower) | (df_features[col] > upper)).sum()
    # Cap (winsorise) instead of drop — preserves row count
    df_features[col] = df_features[col].clip(lower=lower, upper=upper)
    if outliers > 0:
        print(f'{col}: {outliers} outliers capped  (lower={lower:.1f}, upper={upper:.1f})')

In [ ]:
# --- Boxplots AFTER handling ---
numeric_cols = df_features.select_dtypes(include='number').columns.tolist()
if len(numeric_cols) >= 3:
    sample_cols = numeric_cols[:3]
else:
    sample_cols = numeric_cols

fig, axes = plt.subplots(1, len(sample_cols), figsize=(15, 4))
if len(sample_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, sample_cols):
    sns.boxplot(y=df_features[col], ax=ax, color='seagreen')
    ax.set_title(f'{col} — After')
plt.suptitle('Boxplots After Outlier Capping')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'boxplots_after_outlier_capping.png', dpi=150)
plt.show()

## Step 4 : Encode Categoricals

In [ ]:
le = LabelEncoder()

# M1 Dataset categorical columns to encode
cat_cols = df_features.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns to encode: {cat_cols}')

# Binary encoding for Yes/No columns only (not user_pcmac — that is device type)
binary_map = {'Yes': 1, 'No': 0}
yes_no_cols = ['trust_apple', 'familiarity_m1']
for col in yes_no_cols:
    if col in df_features.columns:
        df_features[col] = df_features[col].map(binary_map)
        print(f'Encoded {col} with binary map')

# user_pcmac: Apple vs PC/Others (also accepts Yes/No from forms: Mac/PC)
if 'user_pcmac' in df_features.columns:
    pcmac_norm = df_features['user_pcmac'].astype(str).str.strip().str.capitalize()
    pcmac_map = {'Apple': 1, 'Pc': 0, 'Hp': 0, 'Other': 0, 'Yes': 1, 'No': 0}
    df_features['user_pcmac'] = pcmac_norm.map(pcmac_map)
    if df_features['user_pcmac'].isnull().any():
        bad = sorted(pcmac_norm[df_features['user_pcmac'].isnull()].unique().tolist())
        raise ValueError(f'Unknown user_pcmac values after normalize: {bad}')
    print('Encoded user_pcmac (Apple/Yes=1, PC or other/No=0)')

# Label encode remaining categorical columns (gender, status, domain, etc.)
for col in df_features.select_dtypes(include='object').columns:
    if col not in yes_no_cols:
        df_features[col] = le.fit_transform(df_features[col].astype(str))
        print(f'Label encoded {col}')

print('\nDtypes after encoding:')
print(df_features.dtypes)
df_features.head()

In [ ]:
# Diagnostic: inspect raw df and df_features values after encoding
print('Raw df sample (first 5 rows):')
print(df.head())
print('\nProcessed feature DF (first 5 rows):')
print(df_features.head())

print('\nEncoded categorical columns sample values:')
for col in ['trust_apple', 'user_pcmac', 'familiarity_m1', 'gender', 'status', 'domain']:
    if col in df_features.columns:
        print(f"  {col}: {pd.Series(df_features[col]).unique()[:10]}")

## Step 5 : Feature Scaling (StandardScaler)

In [ ]:
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_features), columns=df_features.columns)

# Add the target variable back (unscaled) for classification training
if 'm1_purchase' in locals():
    df_scaled['m1_purchase'] = y.reset_index(drop=True).values
    print("✓ Target variable (m1_purchase) added to scaled data")

print('\nScaled stats (mean ≈ 0, std ≈ 1):')
print(df_scaled.describe().round(2))
print(f'\nShape: {df_scaled.shape}')

## Correlation Heatmap (after processing)

In [ ]:
plt.figure(figsize=(12, 10))
# Correlation of features only (exclude target if present)
df_for_corr = df_scaled.drop(columns=['m1_purchase'], errors='ignore')
sns.heatmap(df_for_corr.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Heatmap (Processed M1 Data)')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'correlation_heatmap_processed_m1_data.png', dpi=150)
plt.show()

## Save Processed Data

In [ ]:
# Save processed data with target variable
df_scaled['m1_purchase'] = y.map({'Yes': 1, 'No': 0}).values
output_path = '../data/processed/processed_m1_data.csv'
df_scaled.to_csv(output_path, index=False)
print(f'✓ Saved processed_m1_data.csv — shape: {df_scaled.shape}')
print(f'  Columns ({len(df_scaled.columns)}): {list(df_scaled.columns)}')
print(f'  m1_purchase distribution: {df_scaled["m1_purchase"].value_counts().to_dict()}')

# Save preprocessing artifacts for API inference
import pickle
import os

models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)

# Save scaler
with open(os.path.join(models_dir, 'scaler_m1.pkl'), 'wb') as f:
    pickle.dump(scaler, f)
print(f'✓ Saved scaler_m1.pkl')

# Save preprocessing config for M1 dataset
preprocess_config = {
    'binary_map': {'Yes': 1, 'No': 0},
    'user_pcmac_map': {'Apple': 1, 'Pc': 0, 'Hp': 0, 'Other': 0, 'Yes': 1, 'No': 0},
    'label_encoders': {},  # Store encoders for categorical columns if needed
}

with open(os.path.join(models_dir, 'encoders_m1.pkl'), 'wb') as f:
    pickle.dump(preprocess_config, f)
print(f'✓ Saved encoders_m1.pkl with preprocessing config')